In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q22/annotated/checkpoints/pre_cell_2.pickle

trying: ['load_customer']
me:  1
trying: ['STORAGE_OPTS']
me:  0
trying: ['tpch_parent']
me:  0
trying: ['DATA_ROOT']
me:  0
trying: ['load_orders']
me:  2
trying: ['orders']


me:  2
trying: ['pd']
me:  0
trying: ['customer']
me:  1
Declaring variable STORAGE_OPTS
Declaring variable tpch_parent
Declaring variable DATA_ROOT
Declaring variable pd
Declaring variable load_customer
Declaring variable customer
Declaring variable load_orders
Declaring variable orders


In [4]:
%%RecordEvent
%%time
### cell 2 ###

# Select necessary columns and compute country code
customer_filtered = customer[['C_ACCTBAL', 'C_CUSTKEY']].assign(
    CNTRYCODE=customer['C_PHONE'].str.slice(0, 2)
)

# Filter by positive balance and desired country codes
valid_codes = ['13', '31', '23', '29', '30', '18', '17']
mask = (
    (customer_filtered['C_ACCTBAL'] > 0) &
    customer_filtered['CNTRYCODE'].isin(valid_codes)
)
customer_filtered = customer_filtered[mask]

# Compute average balance and filter above average
avg_value = customer_filtered['C_ACCTBAL'].mean()
customer_filtered = customer_filtered[customer_filtered['C_ACCTBAL'] > avg_value]

# Exclude customers who have placed orders using a GPU-friendly isin filter
customer_selected = customer_filtered[~customer_filtered['C_CUSTKEY'].isin(orders['O_CUSTKEY'])]

# Keep only the columns needed for aggregation
customer_selected = customer_selected[['CNTRYCODE', 'C_ACCTBAL']]

# Compute number of customers per country (GPU groupby.size)
agg1 = (
    customer_selected
    .groupby('CNTRYCODE', as_index=False)
    .size()
    .rename(columns={'size': 'NUMCUST'})
)

# Compute total account balance per country (GPU groupby.sum)
agg2 = (
    customer_selected
    .groupby('CNTRYCODE', as_index=False)['C_ACCTBAL']
    .sum()
    .rename(columns={'C_ACCTBAL': 'TOTACCTBAL'})
)

# Join the two aggregations and sort by country code
total = agg1.merge(agg2, on='CNTRYCODE').sort_values('CNTRYCODE')

CPU times: user 85.6 ms, sys: 31.5 ms, total: 117 ms
Wall time: 117 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q22/rewritten/o4_mini_high/checkpoints/post_cell_2_try_4.pickle

migration speed (bps): 799081539.5889924
---------------------------
variables to migrate:
DATA_ROOT 80
tpch_parent 54
agg1 48
agg2 48
pd 72
customer_filtered 48
valid_codes 120
avg_value 32
orders 48
mask 48
customer_selected 48
STORAGE_OPTS 64
load_customer 144
customer 48
total 48
load_orders 144
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q22/rewritten/o4_mini_high/checkpoints/post_cell_2_try_4.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['customer']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['orders']
Future variables ['customer']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['agg1', 'customer_filtered', 'valid_codes', 'customer_selected', 'agg2', 'mask', 'total', 'avg_value']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q22/opt_cell_exec_info_2_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[2], f)


In [8]:
opt_output = Out.get(4)

In [9]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q22/annotated/checkpoints/post_cell_2.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['customer_selected']
me:  3
trying: ['orders_filtered']
me:  3
trying: ['pd']
me:  0
trying: ['agg2']
me:  3
trying: ['orders']


me:  2
trying: ['agg1']
me:  3
trying: ['avg_value']
me:  3
trying: ['STORAGE_OPTS']
me:  0
trying: ['load_customer']
me:  1
trying: ['total']
me:  3
trying: ['customer']
me:  1
trying: ['customer_keys']
me:  3
trying: ['load_orders']
me:  2
trying: ['orig_output']
me:  4
trying: ['customer_filtered']
me:  3
trying: ['DATA_ROOT']
me:  0
trying: ['tpch_parent']
me:  0
Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable DATA_ROOT
Declaring variable tpch_parent
Declaring variable load_customer
Declaring variable customer
Declaring variable orders
Declaring variable load_orders
Declaring variable customer_selected
Declaring variable orders_filtered
Declaring variable agg2
Declaring variable agg1
Declaring variable avg_value
Declaring variable total
Declaring variable customer_keys
Declaring variable customer_filtered
Declaring variable orig_output
